In [5]:
import PIL.Image
import numpy as np

In [6]:
class CellCounter:
    def __init__(self, image_path, z_upper_threshold, z_lower_threshold, degree_threshold, epsilon=1) -> None:
        # Reading raw image and resizing to 480x360 which preserves the 16:12 aspect ratio
        self.raw_image = PIL.Image.open(image_path).resize((480, 360), resample=PIL.Image.Resampling.BILINEAR)

        # converting to array, computing mean and std of Grayscale values for z_scores
        img_array = np.asarray(self.raw_image)
        color_mean = img_array.mean()
        color_stdev = img_array.std()

        # removing pixels with grayscale z_scores less than the z_threshold. I.e. we only keep the extreme colors and the rest is set to 0
        self.filtered_img_array = np.multiply(img_array, np.bitwise_or(img_array > color_mean + z_upper_threshold*color_stdev,  img_array < color_mean - z_lower_threshold*color_stdev).astype(int))

        # computing distance matrix and storing list of nonzero coordinates
        distance_matrix = self.get_distance_matrix(self.filtered_img_array)

        # clustering, note that the cluster method returns the cluster count but also stores the clustered image as a property of the CellCounter object
        self.cluster_count = self.cluster(distance_matrix, epsilon, degree_threshold)
        self.percentage_differentiated = np.sum((self.clustered_image > 0).astype(int)) / (480*360)


    def get_distance_matrix(self, img_array) -> np.array:
        # getting nonzero pixel locations as boolean matrix
        non_zero = (img_array  > 0).astype(int)
        
        nonzero_coords = []
        # for each nonzero pixel, get the row and column as the x and y val respectively and append to our list of coordinates
        for row, i in enumerate(non_zero):
            for col, val in enumerate(i):
                if val > 0:
                    nonzero_coords.append((row,col))

        # converting list to matrix
        self.nonzero_coords = np.array(nonzero_coords)
        print(len(nonzero_coords))

        # computing taxi cab distances. Note I am using numpys broadcasting feature so all looping is pushed to the numpy backend
        distance_matrix = np.sum(np.abs(self.nonzero_coords.reshape(1, self.nonzero_coords.shape[0], self.nonzero_coords.shape[1]) - self.nonzero_coords.reshape(self.nonzero_coords.shape[0], 1, self.nonzero_coords.shape[1])), axis = 2)
    
        return distance_matrix

    def cluster(self, distance_matrix, epsilon, degree_threshold) -> int:
        # compute adjacency matrix using epsilon as a distance threshold for considering two points adjacent
        adjacency_matrix = ((distance_matrix > 0) & (distance_matrix <= epsilon)).astype(int)
        # getting degree of each nonzero point
        degree_vector = np.sum(adjacency_matrix, axis = 0)

        # copying filtered image and removing all pixels with degree < degree threshold (this gets rid of single pixels and visual fuzz)
        self.clustered_image = np.copy(self.filtered_img_array)
        for i, point in enumerate(self.nonzero_coords):
            if degree_vector[i] < degree_threshold:
                self.clustered_image[point[0], point[1]] = 0

        # recalculating distances
        noiseless_distance_matrix = self.get_distance_matrix(self.clustered_image)

        # reforming adjacency_matrix, note that we use the same epsilon meaning that we remove no additional points here
        noiseless_adjacency_matrix = ((noiseless_distance_matrix > 0) & (noiseless_distance_matrix <= epsilon)).astype(int)

        # recalculating degree vector and matrix
        degree_vector = np.sum(noiseless_adjacency_matrix, axis = 0)
        degree_matrix = np.diag(degree_vector)

        # computing graph laplacian
        laplacian = degree_matrix - noiseless_adjacency_matrix

        # computing eigenvalues of the laplacian
        eigenvalues = np.linalg.eigvalsh(laplacian)

        # counting the multiplicity of zero as an eigenvalue. Note that the eigenvals are floats so may not be exactly 0 which is why I use a tolerance of 10^-6
        cluster_count = np.count_nonzero(abs(eigenvalues) < 10**-6)
        return cluster_count
    
    def show_images(self, save=True) -> None:
        # this method will display the raw resized image, the filtered image, and the clustered image respectively
        self.raw_image.show()
        self.filtered_img = PIL.Image.fromarray(self.filtered_img_array.astype(np.uint8))
        self.clustered_img = PIL.Image.fromarray(self.clustered_image.astype(np.uint8))
        self.filtered_img.show()
        self.clustered_img.show()

        if save:
            self.raw_image.save('adi-well1-42_RESIZED.png')
            self.filtered_img.save('adi-well1-42_Z_FILTERED.png')
            self.clustered_img.save('adi-well1-42_CLUSTERED.png')


In [7]:
counter = CellCounter('ZOE_ALL/adi-well-1/adi-well1-42.tif', z_upper_threshold=2.0, z_lower_threshold=3, degree_threshold=6, epsilon=5)

2413
1890


In [8]:
counter.show_images(save=True)
print('Number of Clusters: ', counter.cluster_count)
print('Percent Differentiated: ', counter.percentage_differentiated * 100)

Number of Clusters:  55
Percent Differentiated:  1.09375
